In [ ]:
################################################################
#Lista todos os datasets da ANEEL (somente o título)
################################################################

import pandas as pd
import requests
import os

# URL da API do Portal de Dados Abertos da ANEEL
v_url = "https://dadosabertos.aneel.gov.br/api/3/action/package_list"

response = requests.get(v_url)
if response.status_code == 200:
  # Define apenas o nome da pasta e o arquivo
  v_pasta = "datasets"
  v_arquivo = "datasets_aneel.xlsx"

  # Cria a pasta automaticamente se ela ainda não existir
  os.makedirs(v_pasta, exist_ok=True)

  # Combina a pasta com o arquivo
  v_caminho_completo = os.path.join(v_pasta, v_arquivo)

  # armazena lista de datasest
  v_datasets = response.json()["result"]
  # Colocando os resultados (nomes dos datasets) em linhas dentro de um DataFrame
  df_datasets = pd.DataFrame(v_datasets, columns=["Dataset"])

  df_datasets.to_excel(v_caminho_completo, index=False)

  #print(f"Total de datasets encontrados: {len(df_datasets)}")
  #print("\nPrimeiras linhas do DataFrame:")
  #print(df_datasets.head())
  #print(df_datasets)

In [ ]:
################################################################
#Lista todos os datasets da ANEEL com endereço
################################################################

import pandas as pd
import requests
import time

# 1. Obter a lista de todos os nomes de datasets
base_url = "https://dadosabertos.aneel.gov.br/api/3/action"
response = requests.get(f"{base_url}/package_list")
datasets = response.json()["result"]

lista_detalhes = []

# 2. Iterar sobre os datasets para buscar os detalhes (incluindo URL)
# Nota: Limitei a 10 para o script não demorar muito, remova o [:10] para todos
for name in datasets[:100]: 
    try:
        detalhes = requests.get(f"{base_url}/package_show?id={name}").json()["result"]
        lista_detalhes.append({
            "Dataset": detalhes["title"],
            "URL": f"https://dadosabertos.aneel.gov.br/dataset/{detalhes['name']}"
        })
    except Exception as e:
        print(f"Erro ao buscar {name}: {e}")
    
    # Pausa breve para não sobrecarregar a API
    time.sleep(0.5)

# 3. Criar o DataFrame com os resultados
df_datasets = pd.DataFrame(lista_detalhes)

print(df_datasets.head())

if response.status_code == 200:
  # Define apenas o nome da pasta e o arquivo
  v_pasta = "datasets"
  v_arquivo = "datasets_aneel_endereco.xlsx"

  # Cria a pasta automaticamente se ela ainda não existir
  os.makedirs(v_pasta, exist_ok=True)

  # Combina a pasta com o arquivo
  v_caminho_completo = os.path.join(v_pasta, v_arquivo)

df_datasets.to_excel(v_caminho_completo, index=False)

                                             Dataset  \
0              Acréscimo anual da potência instalada   
1             Agentes de Geração de Energia Elétrica   
2                          Agentes do Setor Elétrico   
3  Atendimento a pedidos de conexões MMGD - Mini ...   
4            Atendimento às Ocorrências Emergenciais   

                                                 URL  
0  https://dadosabertos.aneel.gov.br/dataset/acre...  
1  https://dadosabertos.aneel.gov.br/dataset/agen...  
2  https://dadosabertos.aneel.gov.br/dataset/agen...  
3  https://dadosabertos.aneel.gov.br/dataset/aten...  
4  https://dadosabertos.aneel.gov.br/dataset/aten...  


In [ ]:
################################################################
# Lista o conteúdo do link de Bandeiras Tarifárias (somente o título)
################################################################

import os
import pandas as pd
import requests

# URL da API do CKAN para buscar os detalhes do dataset de bandeiras tarifárias
v_url = (
    "https://dadosabertos.aneel.gov.br/api/3/action/package_show?id=bandeiras-tarifarias"
)

response = requests.get(v_url)

if response.status_code == 200:
  # O retorno da API do CKAN vem dentro da chave 'result'
  dados_json = response.json()

  if dados_json.get("success"):
    dataset_info = dados_json["result"]
    recursos = dataset_info.get("resources", [])

    # Colocando os recursos (arquivos disponíveis no dataset) em um DataFrame
    df_recursos = pd.DataFrame(recursos)

    # Define a pasta e o arquivo de saída
    v_pasta = "datasets"
    v_arquivo = "bandeiras_tarifarias_recursos.xlsx"
    os.makedirs(v_pasta, exist_ok=True)
    v_caminho_completo = os.path.join(v_pasta, v_arquivo)

    # Salva em Excel
    df_recursos.to_excel(v_caminho_completo, index=False)

    print(
        f"Sucesso! {len(df_recursos)} recursos encontrados e salvos em"
        f" {v_caminho_completo}"
    )
    print(df_recursos[["name", "format", "url"]])
  else:
    print("A API retornou sucesso falso para o dataset solicitado.")
else:
  print(f"Erro na requisição HTTP: {response.status_code}")

Sucesso! 6 recursos encontrados e salvos em datasets/bandeiras_tarifarias_recursos.xlsx
                                   name format  \
0       Dicionário de dados - Adicional    PDF   
1        Bandeira Tarifária - Adicional    CSV   
2     Dicionário de dados - Acionamento    PDF   
3      Bandeira Tarifária - Acionamento    CSV   
4  Dicionário de Dados - Conta Bandeira    PDF   
5   Bandeira Tarifária - Conta Bandeira    CSV   

                                                 url  
0  https://dadosabertos.aneel.gov.br/dataset/7f43...  
1  https://dadosabertos.aneel.gov.br/dataset/7f43...  
2  https://dadosabertos.aneel.gov.br/dataset/7f43...  
3  https://dadosabertos.aneel.gov.br/dataset/7f43...  
4  https://dadosabertos.aneel.gov.br/dataset/7f43...  
5  https://dadosabertos.aneel.gov.br/dataset/7f43...  


In [ ]:
################################################################
# Bandeiras Tarifárias (Oficial)
################################################################

import os
import pandas as pd
import requests

# 1. Obter a lista de recursos do dataset
api_url = "https://dadosabertos.aneel.gov.br/api/3/action/package_show?id=bandeiras-tarifarias"
response = requests.get(api_url)

if response.status_code == 200:
    dados_json = response.json()["result"]
    recursos = dados_json.get("resources", [])
    
    # Cria a pasta de destino
    v_pasta = "datasets_bandeiras"
    os.makedirs(v_pasta, exist_ok=True)
    
    print(f"Encontrados {len(recursos)} recursos. Iniciando download...")

    # 2. Iterar sobre cada recurso e baixar/salvar
    for item in recursos:
        nome_arquivo = item.get("name")
        url_download = item.get("url")
        formato = item.get("format", "").lower()
        
        # Limpar o nome do arquivo para evitar caracteres inválidos no SO
        nome_limpo = "".join([c if c.isalnum() else "_" for c in nome_arquivo])
        
        try:
            print(f"Baixando: {nome_arquivo} ({formato})...")
            
            # Tentar ler o arquivo diretamente com Pandas
            if formato == 'csv':
                df = pd.read_csv(url_download, sep=';', encoding='latin1')
                df.to_excel(f"{v_pasta}/{nome_limpo}.xlsx", index=False)
            elif formato in ['xls', 'xlsx']:
                df = pd.read_excel(url_download)
                df.to_excel(f"{v_pasta}/{nome_limpo}.xlsx", index=False)
            else:
                # Caso seja outro formato, salva o conteúdo bruto
                resp = requests.get(url_download)
                with open(f"{v_pasta}/{nome_limpo}.{formato}", "wb") as f:
                    f.write(resp.content)
            
            print(f"-> Salvo com sucesso!")
            
        except Exception as e:
            print(f"-> Erro ao baixar {nome_arquivo}: {e}")

else:
    print(f"Erro ao acessar API: {response.status_code}")

Encontrados 6 recursos. Iniciando download...
Baixando: Dicionário de dados - Adicional (pdf)...
-> Salvo com sucesso!
Baixando: Bandeira Tarifária - Adicional (csv)...
-> Salvo com sucesso!
Baixando: Dicionário de dados - Acionamento (pdf)...
-> Salvo com sucesso!
Baixando: Bandeira Tarifária - Acionamento (csv)...
-> Salvo com sucesso!
Baixando: Dicionário de Dados - Conta Bandeira (pdf)...
-> Salvo com sucesso!
Baixando: Bandeira Tarifária - Conta Bandeira (csv)...
-> Salvo com sucesso!


In [4]:
################################################################
# Tarifas (Oficial)
################################################################

import os
import pandas as pd
import requests

# 1. Obter a lista de recursos do dataset
api_url = "https://dadosabertos.aneel.gov.br/api/3/action/package_show?id=tarifas-distribuidoras-energia-eletrica"
response = requests.get(api_url)

if response.status_code == 200:
    dados_json = response.json()["result"]
    recursos = dados_json.get("resources", [])
    
    # Cria a pasta de destino
    v_pasta = "datasets_tarifas"
    os.makedirs(v_pasta, exist_ok=True)
    
    print(f"Encontrados {len(recursos)} recursos. Iniciando download...")

    # 2. Iterar sobre cada recurso e baixar/salvar
    for item in recursos:
        nome_arquivo = item.get("name")
        url_download = item.get("url")
        formato = item.get("format", "").lower()
        
        # Limpar o nome do arquivo para evitar caracteres inválidos no SO
        nome_limpo = "".join([c if c.isalnum() else "_" for c in nome_arquivo])
        
        try:
            print(f"Baixando: {nome_arquivo} ({formato})...")
            
            # Tentar ler o arquivo diretamente com Pandas
            if formato == 'csv':
                df = pd.read_csv(url_download, sep=';', encoding='latin1')
                df.to_excel(f"{v_pasta}/{nome_limpo}.xlsx", index=False)
            elif formato in ['xls', 'xlsx']:
                df = pd.read_excel(url_download)
                df.to_excel(f"{v_pasta}/{nome_limpo}.xlsx", index=False)
            else:
                # Caso seja outro formato, salva o conteúdo bruto
                resp = requests.get(url_download)
                with open(f"{v_pasta}/{nome_limpo}.{formato}", "wb") as f:
                    f.write(resp.content)
            
            print(f"-> Salvo com sucesso!")
            
        except Exception as e:
            print(f"-> Erro ao baixar {nome_arquivo}: {e}")

else:
    print(f"Erro ao acessar API: {response.status_code}")

Encontrados 3 recursos. Iniciando download...
Baixando: Dicionário de dados (pdf)...
-> Salvo com sucesso!
Baixando: tarifas-homologadas-distribuidoras-energia-eletrica.csv (csv)...
-> Salvo com sucesso!
Baixando: tarifas-homologadas-distribuidoras-energia-eletrica.xml (xml)...
-> Salvo com sucesso!


In [5]:
################################################################
# Empreendimento (Oficial)
################################################################

import os
import pandas as pd
import requests

# 1. Obter a lista de recursos do dataset
api_url = "https://dadosabertos.aneel.gov.br/api/3/action/package_show?id=relacao-de-empreendimentos-de-geracao-distribuida"
response = requests.get(api_url)

if response.status_code == 200:
    dados_json = response.json()["result"]
    recursos = dados_json.get("resources", [])
    
    # Cria a pasta de destino
    v_pasta = "datasets_empreendimentos"
    os.makedirs(v_pasta, exist_ok=True)
    
    print(f"Encontrados {len(recursos)} recursos. Iniciando download...")

    # 2. Iterar sobre cada recurso e baixar/salvar
    for item in recursos:
        nome_arquivo = item.get("name")
        url_download = item.get("url")
        formato = item.get("format", "").lower()
        
        # Limpar o nome do arquivo para evitar caracteres inválidos no SO
        nome_limpo = "".join([c if c.isalnum() else "_" for c in nome_arquivo])
        
        try:
            print(f"Baixando: {nome_arquivo} ({formato})...")
            
            # Tentar ler o arquivo diretamente com Pandas
            if formato == 'csv':
                df = pd.read_csv(url_download, sep=';', encoding='latin1')
                df.to_excel(f"{v_pasta}/{nome_limpo}.xlsx", index=False)
            elif formato in ['xls', 'xlsx']:
                df = pd.read_excel(url_download)
                df.to_excel(f"{v_pasta}/{nome_limpo}.xlsx", index=False)
            else:
                # Caso seja outro formato, salva o conteúdo bruto
                resp = requests.get(url_download)
                with open(f"{v_pasta}/{nome_limpo}.{formato}", "wb") as f:
                    f.write(resp.content)
            
            print(f"-> Salvo com sucesso!")
            
        except Exception as e:
            print(f"-> Erro ao baixar {nome_arquivo}: {e}")

else:
    print(f"Erro ao acessar API: {response.status_code}")

Encontrados 12 recursos. Iniciando download...
Baixando: Dicionário de dados (pdf)...
-> Salvo com sucesso!
Baixando: empreendimento-geracao-distribuida.zip (zip)...
-> Salvo com sucesso!
Baixando: empreendimento-geracao-distribuida.parquet (parquet)...
-> Salvo com sucesso!
Baixando: DM - Empreendimentos Geração Distribuída Eólica - Informações Técnicas.pdf (pdf)...
-> Salvo com sucesso!
Baixando: empreendimento-gd-informacoes-tecnicas-eolica.csv (csv)...
-> Salvo com sucesso!
Baixando: DM - Empreendimentos Geração Distribuída Fotovoltaica - Informações Técnicas.pdf (pdf)...
-> Salvo com sucesso!
Baixando: empreendimento-gd-informacoes-tecnicas-fotovoltaica.zip (zip)...
-> Salvo com sucesso!
Baixando: DM - Empreendimentos Geração Distribuída Hidrelétrica - Informações Técnicas.pdf (pdf)...
-> Salvo com sucesso!
Baixando: empreendimento-gd-informacoes-tecnicas-hidreletrica.csv (csv)...
-> Salvo com sucesso!
Baixando: DM - Empreendimentos Geração Distribuída Termelétrica - Informações T